In [1]:
import json

from pymongo import MongoClient

# Load data from JSONL file
client = MongoClient("mongodb://localhost:27017/")
db = client["ddxplus"]
collection_train = db["train-semistructured"]
collection_val = db["validate-semistructured"]
collection_test = db["test-semistructured"]

train_data = list(
    collection_train.find(
        {},
        {
            "_id": 0,
            "PATHOLOGY": 0,
            "EVIDENCES": 0,
            "EVIDENCES_JSON_V2": 0,
            "DIFFERENTIAL_DIAGNOSIS": 0,
        },
    )
)
val_data = list(
    collection_val.find(
        {},
        {
            "_id": 0,
            "PATHOLOGY": 0,
            "EVIDENCES": 0,
            "EVIDENCES_JSON_V2": 0,
            "DIFFERENTIAL_DIAGNOSIS": 0,
        },
    )
)
test_data = list(
    collection_test.find(
        {},
        {
            "_id": 0,
            "PATHOLOGY": 0,
            "EVIDENCES": 0,
            "EVIDENCES_JSON_V2": 0,
            "DIFFERENTIAL_DIAGNOSIS": 0,
        },
    )
)

print(f"Train samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Test samples: {len(test_data)}")

# show one train sample
print("Example train sample:")
print(json.dumps(train_data[0], indent=2))

TARGET_KEY = "DIFFERENTIAL_DIAGNOSIS_NOPROB"

Train samples: 1025602
Validation samples: 132448
Test samples: 134529
Example train sample:
{
  "AGE": 18,
  "SEX": "M",
  "INITIAL_EVIDENCE": "E_91",
  "EVIDENCES_JSON_V1": {
    "E_48": [],
    "E_50": [],
    "E_53": [],
    "E_54": [
      "V_161",
      "V_183"
    ],
    "E_55": [
      "V_89",
      "V_108",
      "V_167"
    ],
    "E_56": [
      "4"
    ],
    "E_57": [
      "V_123"
    ],
    "E_58": [
      "3"
    ],
    "E_59": [
      "3"
    ],
    "E_77": [],
    "E_79": [],
    "E_91": [],
    "E_97": [],
    "E_201": [],
    "E_204": [
      "V_10"
    ],
    "E_222": []
  },
  "DIFFERENTIAL_DIAGNOSIS_NOPROB": [
    "Bronchitis",
    "Pneumonia",
    "URTI",
    "Bronchiectasis",
    "Tuberculosis",
    "Influenza",
    "HIV (initial infection)",
    "Chagas"
  ]
}


In [ ]:
import random

import torch

from origami import DataConfig, ModelConfig, OrigamiConfig, OrigamiPipeline, TrainingConfig
from origami.training import TableLogCallback, array_f1, array_jaccard

# For reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

config = OrigamiConfig(
    data=DataConfig(
        numeric_mode="none",
    ),
    model=ModelConfig(
        d_model=256,
        n_layers=6,
        n_heads=8,
    ),
    training=TrainingConfig(
        learning_rate=0.001,
        eval_strategy="steps",
        eval_steps=100,
        eval_sample_size=100,
        eval_metrics={"jaccard": array_jaccard, "f1": array_f1},
        shuffle_keys=True,
        batch_size=100,
        target_key=TARGET_KEY,
    ),
)

pipeline = OrigamiPipeline(config)
callback = TableLogCallback(print_every=10)

In [ ]:
pipeline.fit(train_data, eval_data=val_data, callbacks=[callback], epochs=4, verbose=True)

In [ ]:
pipeline.save("ddxplus_origami_pipeline.pt")

In [2]:
from origami import OrigamiPipeline

pipeline = OrigamiPipeline.load("ddxplus_origami_pipeline.pt")

In [ ]:
from origami.training import array_f1, array_precision, array_recall

pipeline.evaluate(
    test_data,
    metrics={"f1": array_f1, "precision": array_precision, "recall": array_recall},
    batch_size=256,
    verbose=True,
)

In [ ]:
pipeline.predict_batch(
    test_data[:1000],
    target_key=TARGET_KEY,
    allow_complex_values=True,
    batch_size=256,
    profile=True,
)

AttributeError: 'OrigamiModel' object has no attribute 'device'